<a href="https://colab.research.google.com/github/mejia080902-bit/simulacion_II/blob/main/simulacion_II_variables_de_control.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Simulación II - Comparación de eficiencia:
Monte Carlo Crudo vs Variables de Control
Estimamos I = E[g(X)], con X ~ N(0,1) y g(x) = 1/(1+x^2)
"""

import numpy as np
import time

# -------------------------
# Función de interés
# -------------------------
def g(x):
    # g(x) = 1 / (1 + x^2)
    return 1.0 / (1.0 + x**2)

# -------------------------
# Método Monte Carlo Crudo
# -------------------------
def crudo(N, rng=None):
    """
    Retorna: media, varianza muestral (unbiased), desv. est., lista de g(x)
    """
    if rng is None:
        rng = np.random.default_rng()
    x = rng.normal(loc=0.0, scale=1.0, size=N)
    gx = g(x)
    mean = gx.mean()
    var = gx.var(ddof=1)
    std = var**0.5
    return mean, var, std, gx

# -------------------------
# Método de Variables de Control
# -------------------------
def variables_de_control(N, rng=None):
    """
    Control variate: h(x) = x^2, E[h] = 1 para X~N(0,1).
    Estimador: mean_g - alpha*(mean_h - Eh) con alpha* = Cov(g,h)/Var(h).
    Retorna: estimador, varianza muestral del estimador, desv. est., dict con detalles.
    """
    if rng is None:
        rng = np.random.default_rng()
    x = rng.normal(loc=0.0, scale=1.0, size=N)
    gx = g(x)
    hx = x**2
    Eh = 1.0  # E[X^2] = 1 para N(0,1)

    # Estadísticos muestrales
    mean_g = gx.mean()
    mean_h = hx.mean()
    cov_gh = np.cov(gx, hx, ddof=1)[0, 1]
    var_h = hx.var(ddof=1)

    # Alpha óptimo y estimador CV
    alpha = cov_gh / var_h
    est = mean_g - alpha * (mean_h - Eh)

    # Varianza del estimador CV (aprox asintótica por delta method)
    var_g = gx.var(ddof=1)
    var_est = var_g + alpha**2 * var_h - 2 * alpha * cov_gh
    std_est = var_est**0.5

    info = {
        "alpha_hat": alpha,
        "mean_g": mean_g,
        "mean_h": mean_h,
        "cov_gh": cov_gh,
        "var_h": var_h,
        "rho": cov_gh / (gx.std(ddof=1) * hx.std(ddof=1) + 1e-12),  # correlación
    }
    return est, var_est, std_est, info

# -------------------------
# Ejecución y tiempos
# -------------------------
if __name__ == "__main__":
    N = 10_000  # número de simulaciones
    rng = np.random.default_rng(12345)  # reproducible

    # Crudo
    t0 = time.time()
    mean_crudo, var_crudo, std_crudo, gx = crudo(N, rng)
    t_crudo = time.time() - t0

    # Variables de control
    t0 = time.time()
    est_cv, var_cv, std_cv, info = variables_de_control(N, rng)
    t_cv = time.time() - t0

    # -------------------------
    # Resultados
    # -------------------------
    print("---- Método Crudo ----")
    print(f"Estimador = {mean_crudo:.6f}, Var = {var_crudo:.6e}, Std = {std_crudo:.6e}, Tiempo = {t_crudo:.5f} s")

    print("\n---- Método Variables de Control ----")
    print(f"Estimador = {est_cv:.6f}, Var = {var_cv:.6e}, Std = {std_cv:.6e}, Tiempo = {t_cv:.5f} s")
    print(f"alpha* = {info['alpha_hat']:.6f}, rho(g,h) = {info['rho']:.6f}")

    # -------------------------
    # Comparación de eficiencia (costo-tiempo ajustado)
    # Métrica: ε = (t * Var) método / (t * Var) crudo
    # ε < 1  => método en el numerador más eficiente
    # -------------------------
    eps_cv = (t_cv * var_cv) / (t_crudo * var_crudo)
    print(f"\nEficiencia relativa ε(CV vs Crudo) = {eps_cv:.6f}")
    if eps_cv < 1:
        print("→ Variables de control es más eficiente que crudo.")
    else:
        print("→ Crudo es más eficiente que variables de control.")

    # (Opcional) error estándar del estimador de la media:
    se_crudo = (var_crudo / N) ** 0.5
    se_cv = (var_cv / N) ** 0.5
    print(f"\nError estándar de la media (Crudo): {se_crudo:.6e}")
    print(f"Error estándar de la media (CV)   : {se_cv:.6e}")


---- Método Crudo ----
Estimador = 0.657259, Var = 7.157935e-02, Std = 2.675432e-01, Tiempo = 0.00058 s

---- Método Variables de Control ----
Estimador = 0.655327, Var = 2.200746e-02, Std = 1.483491e-01, Tiempo = 0.00541 s
alpha* = -0.154289, rho(g,h) = -0.827089

Eficiencia relativa ε(CV vs Crudo) = 2.885450
→ Crudo es más eficiente que variables de control.

Error estándar de la media (Crudo): 2.675432e-03
Error estándar de la media (CV)   : 1.483491e-03
